# Intro

- Recoding of https://github.com/Daniel-EST/deep-steganography/ but updated to match modern tensorflow or pytorch depending what is easiest to use
- from paper: 500 training images, 50 validation images, and 50 test images, all uniformly resized to a manageable dimension of 64x64x3 pixels
    - with rbg colour (no greyscale) to ease training

- overall process:
  - get text which is small enough *after* huffman encoding to fit within 64*64*3 sized block, and encode within image using LSB resulting in **secret image** which will be used by NN (for training all will be encoded with *"Hello World!"* just to keep things simple, but randomized text could be more authentic)
    - LSB - use pillow library (https://pypi.org/project/pillow/) to ease implementation
    - Huffman will also be from GeeksForGeeks / public implementation to speed development
  - NN will be three layers a preperation > hide > reveal networks (with each being NNs) that will ideally allow for full reconstruction of secret image
  - finally undo LSB + Huffman to retrieve final text and check accordingly.

- Issues so far:
  - Deep stenography is greatly outdated so fixing has not been very easy, could promp re-scope of assign to meet deadline if it is unable to be done
  - reformatting between secret image > steg image > revealed secret causes issues with LSB where unless its 100% (>90% ssim) then final classical steganography cannot be performed often
  - could add scheduler to improve flattening learning rate after x epochs and this could benefit the issues with too much noise ruining secret text recovery.

In [45]:
from LSBSteg.LSBSteg import LSBSteg
from dahuffman import HuffmanCodec

# code example from https://github.com/RobinDavid/LSB-Steganography libraru, lsb is using
# LSB 
def LSBEmbed(text,image):
    # Embed text into image using least sig bit method
    steg = LSBSteg(image)
    img_encoded = steg.encode_binary(text)
    return img_encoded

def LSBExtract(image):
    # pull least sig bit from image and reformat into embedded text
    steg = LSBSteg(image)
    extracted_text = steg.decode_binary()
    return extracted_text

# huffman serialization assisted with chatgpt for debugging, dahuffman use from PyPl example
def HuffmanEncode(data):
    # encode text into huffman tree + map
    codec = HuffmanCodec.from_data(data)
    encoded_data = codec.encode(data)

    # finally prepend all so tree:data is one block
    return codec, encoded_data

def HuffmanDecode(codec,data):
    # using Huffman map reconstruct text to original values and decode / unzip data
    decoded_data = codec.decode(data)
    return decoded_data

In [46]:
from datasets import load_dataset
import os, certifi
import numpy as np
import random

os.environ['SSL_CERT_FILE'] = certifi.where()
def is_rgb(example):
    return example['image'].mode == 'RGB'
# option 1 uses "Hello World!" for all to reuse codec and make things simpler
temp_data = load_dataset("zh-plus/tiny-imagenet")
tiny_imageNet_rgb = temp_data.filter(is_rgb)

train_pool = tiny_imageNet_rgb['train']
test_pool = tiny_imageNet_rgb['valid']

ex_data = "Hello World!"
codec,encoded_data = HuffmanEncode(ex_data)

# 2. Sample from Training Pool
num_train = 500
train_indices = random.sample(range(len(train_pool)), num_train * 2)
c_train_imgs = [train_pool[i]['image'] for i in train_indices[:num_train]]
s_train_imgs = [train_pool[i]['image'] for i in train_indices[num_train:]]

# 3. Sample from Test Pool (Audit Set)
num_test = 50
test_indices = random.sample(range(len(test_pool)), num_test * 2)
c_test_imgs = [test_pool[i]['image'] for i in test_indices[:num_test]]
s_test_imgs = [test_pool[i]['image'] for i in test_indices[num_test:]]

s_train_steg = []
for img in s_train_imgs:
    np_img = np.array(img)
    s_train_steg.append( LSBEmbed(encoded_data,np_img))

s_test_steg = []
for img in s_test_imgs:
    np_img = np.array(img)
    s_test_steg.append( LSBEmbed(encoded_data,np_img))

print(f"Pools Ready: {len(c_train_imgs)} Train pairs, {len(c_test_imgs)} Test pairs.")

Pools Ready: 500 Train pairs, 50 Test pairs.


## NN Format etc.
### Input Layer??? (shown in diagram but may not be relevant...)
- IN:
  - secret image: (~,64,64,3)
- OUT:
  - secret image: (~,64,64,3)
### Prep Layer (prep image for embed)
- IN:
  - secret image: (~,64,64,3)
- OUT:
  - prepared secret image (~, 64,64,150)

#### NN Steps
- INIT:
  ```python
    self.conv_layer_4_3x3 = ConvLayer(4, filters=50, kernel_size=(3, 3), activation=tf.nn.relu)
    self.conv_layer_4_4x4 = ConvLayer(4, filters=50, kernel_size=(4, 4), activation=tf.nn.relu)
    self.conv_layer_4_5x5 = ConvLayer(4, filters=50, kernel_size=(5, 5), activation=tf.nn.relu)

    self.concat_1 = tf.keras.layers.Concatenate(axis=3)

    self.conv_1_3x3 = ConvLayer(1, filters=50, kernel_size=(3, 3), activation=tf.nn.relu)
    self.conv_1_4x4 = ConvLayer(1, filters=50, kernel_size=(4, 4), activation=tf.nn.relu)
    self.conv_1_5x5 = ConvLayer(1, filters=50, kernel_size=(5, 5), activation=tf.nn.relu)

    self.concat_2 = tf.keras.layers.Concatenate(axis=3)
  ```
  - what its doing:
    - 2 groups (first 4 channel (batch, h,w,rbg) and second 1 channel (??)) passing to 2d convolution with 50 filters, kernel 3x3 -> 5x5
    - not clear on what 2nd group is for but maybe just more data (so calc between both groups and between 3^2, 4^2, 5^2 matrices allows best quality?)
  
- CALL:
  ```python
    prep_input = tf.keras.layers.Rescaling(1./255, input_shape=input_tensor.shape)(input_tensor)
    conv_4_3x3 = self.conv_layer_4_3x3(prep_input, training=training)
    conv_4_4x4 = self.conv_layer_4_4x4(prep_input, training=training)
    conv_4_5x5 = self.conv_layer_4_5x5(prep_input, training=training)

    concat_1 = self.concat_1([conv_4_3x3, conv_4_4x4, conv_4_5x5])

    conv_1_3x3 =  self.conv_1_3x3(concat_1)
    conv_1_4x4 =  self.conv_1_4x4(concat_1)
    conv_1_5x5 =  self.conv_1_5x5(concat_1)

    return self.concat_2([conv_1_3x3, conv_1_4x4, conv_1_5x5])
  ```
  - what its doing:
    - basically just forward pass for above, so almost same format...
      - scaling 1./255 is interesting but not clear on why but google says normalizes pixels to log 0-1 vs 0-255 so making sure these are kept as tensors I guess.
### Hide Layer (embed prepared secret with cover)
- IN:
  - prepared secret image (~, 64,64,150)
  - cover image (~, 64,64,3)
- OUT:
  - hidden/steg image: (~64,64,3)

#### NN Steps
- IN:
  ```python
    self.prep_layer = PrepLayer()
    self.concat_1 = tf.keras.layers.Concatenate(axis=3)

    self.conv_layer_4_3x3 = ConvLayer(4, filters=50, kernel_size=(3, 3), activation=tf.nn.relu)
    self.conv_layer_4_4x4 = ConvLayer(4, filters=50, kernel_size=(4, 4), activation=tf.nn.relu)
    self.conv_layer_4_5x5 = ConvLayer(4, filters=50, kernel_size=(5, 5), activation=tf.nn.relu)

    self.concat_2 = tf.keras.layers.Concatenate(axis=3)

    self.conv_1_3x3 = ConvLayer(1, filters=50, kernel_size=(3, 3), activation=tf.nn.relu)
    self.conv_1_4x4 = ConvLayer(1, filters=50, kernel_size=(4, 4), activation=tf.nn.relu)
    self.conv_1_5x5 = ConvLayer(1, filters=50, kernel_size=(5, 5), activation=tf.nn.relu)

    self.concat_3 = tf.keras.layers.Concatenate(axis=3)

    self.conv_1_1x1 = ConvLayer(1, filters=3, kernel_size=(1, 1), activation=tf.nn.relu)
  ```
  - what its doing:
    - similar to prep but with final conv after, and calling prep then concat before NN steps?
- CALL:
  ```python
    prep_input = input_tensor[0]
    hide_input = tf.keras.layers.Rescaling(1./255, input_shape=input_tensor[1].shape)(input_tensor[1])
    concat_1 = self.concat_1([prep_input, hide_input])

    conv_4_3x3 = self.conv_layer_4_3x3(concat_1, training=training)
    conv_4_4x4 = self.conv_layer_4_4x4(concat_1, training=training)
    conv_4_5x5 = self.conv_layer_4_5x5(concat_1, training=training)

    concat_2 = self.concat_2([conv_4_3x3, conv_4_4x4, conv_4_5x5])

    conv_1_3x3 =  self.conv_1_3x3(concat_2)
    conv_1_4x4 =  self.conv_1_4x4(concat_2)
    conv_1_5x5 =  self.conv_1_5x5(concat_2)

    concat_3 = self.concat_3([conv_1_3x3, conv_1_4x4, conv_1_5x5])

    return self.conv_1_1x1(concat_3)
  ```
  - what its doing:
    - just forward for hide like with prep...
### Reveal Layer (decode / recover secret)
- IN:
  - hidden/steg image: (~,64,64,3)
- OUT:
  - recovered secret image: (~,64,64,3)
#### NN Steps
- IN:
  ```python
    self.conv_layer_4_3x3 = ConvLayer(4, filters=50, kernel_size=(3, 3), activation=tf.nn.relu)
    self.conv_layer_4_4x4 = ConvLayer(4, filters=50, kernel_size=(4, 4), activation=tf.nn.relu)
    self.conv_layer_4_5x5 = ConvLayer(4, filters=50, kernel_size=(5, 5), activation=tf.nn.relu)

    self.concat_1 = tf.keras.layers.Concatenate(axis=3)

    self.conv_1_3x3 = ConvLayer(1, filters=50, kernel_size=(3, 3), activation=tf.nn.relu)
    self.conv_1_4x4 = ConvLayer(1, filters=50, kernel_size=(4, 4), activation=tf.nn.relu)
    self.conv_1_5x5 = ConvLayer(1, filters=50, kernel_size=(5, 5), activation=tf.nn.relu)

    self.concat_2 = tf.keras.layers.Concatenate(axis=3)

    self.conv_1_1x1 = ConvLayer(1, filters=3, kernel_size=(1, 1), activation=tf.nn.relu)
  ```
  - what its doing:
    - same as prep but using hidden image so this is training to undo... but otherwise the same thing basically
- CALL:
  ```python
    conv_4_3x3 = self.conv_layer_4_3x3(input_tensor, training=training)
    conv_4_4x4 = self.conv_layer_4_4x4(input_tensor, training=training)
    conv_4_5x5 = self.conv_layer_4_5x5(input_tensor, training=training)

    concat_1 = self.concat_1([conv_4_3x3, conv_4_4x4, conv_4_5x5])

    conv_1_3x3 =  self.conv_1_3x3(concat_1)
    conv_1_4x4 =  self.conv_1_4x4(concat_1)
    conv_1_5x5 =  self.conv_1_5x5(concat_1)

    concat_2 = self.concat_2([conv_1_3x3, conv_1_4x4, conv_1_5x5])

    return self.conv_1_1x1(concat_2)
  ```
  - what its doing:
    - forward of above

### Steg Loss
- IN:
  - secret and reveal MSE values / model to derive MSEs
- OUT:
  - loss value in MSE for training
#### Code
- IN:
  ```python
      beta = tf.constant(self.beta, name='beta')

      secret_true = y_true[0]
      secret_pred = y_pred[0]

      cover_true = y_true[1]
      cover_pred = y_pred[1]

      secret_mse = tf.losses.MSE(secret_true, secret_pred)
      cover_mse = tf.losses.MSE(cover_true, cover_pred)

      return tf.reduce_mean(cover_mse + beta * secret_mse)
  ```
  - what its doing:
    - take beta (as constant) and then do cover + secret with reduce mean func to find final MSE, balanced between both. kinda just average mse between secret and reveal not too complex

### Steg Loss
- IN:
  - 
- OUT:
  - 
#### Code
- IN:
  ```python
      
  ```
  - what its doing:
    - 

In [ ]:
# steg NN
# Prep Layer

# Conv Layer

# Reveal Layer

In [ ]:
# --- Training Configuration ---
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 1e-3

print("Training Complete.")

Starting Training on 500 pairs...
Epoch [1/10] | Total Loss: 0.074884 | Cover MSE: 0.031396 | Secret MSE: 0.004352


KeyboardInterrupt: 

In [58]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

print(torch.cuda.is_available())

def extract_with_transform(tensor):
    # 1. Clean the tensor (Detach and Squeeze)
    # ToPILImage does not handle 4D batches automatically
    t = tensor.detach().cpu().squeeze(0) if tensor.dim() == 4 else tensor.detach().cpu()

    # 2. CRITICAL STEP: Manual Rounding
    # We round while still a tensor to ensure 127.9 becomes 128.0
    t = torch.clamp(t * 255.0, 0, 255).round() / 255.0

    # 3. Use Transform for the rest
    # This handles the [C,H,W] -> [H,W,C] swap and uint8 conversion
    to_img = transforms.ToPILImage()
    pil_img = to_img(t)
    
    # Return as numpy array for your LSBSteg library
    return np.array(pil_img)

def calculate_metrics(original, reconstructed):
    # Ensure images are uint8 and same shape
    original = original.astype(np.uint8)
    reconstructed = reconstructed.astype(np.uint8)
    
    psnr_val = psnr(original, reconstructed)
    # For SSIM on color images, set channel_axis=-1 (HWC format)
    ssim_val = ssim(original, reconstructed, channel_axis=-1)
    
    return psnr_val, ssim_val

i=0
for i in range(i,10):
    secret_input = steg_transform(s_test_steg[i]).unsqueeze(0).to(device)
    cover_input = img_transform(c_test_imgs[i]).unsqueeze(0).to(device)

    # 2. Model Inference
    model.eval()
    with torch.no_grad():
        container, revealed = model(secret_input, cover_input)
    rev_img = extract_from_tensor(revealed)
    sec_img = extract_from_tensor(secret_input)
    psnr_val, ssim_val = calculate_metrics(sec_img,rev_img)
    # print(sec_img,rev_img,'\n\n\n')
    print(f'PSNR: {psnr_val}, SSIM: {ssim_val}')
    # rev_img = extract_from_tensor(rev_img)
    # print(rev_img)
    extracted_code = LSBExtract(rev_img)
    print(extracted_code)
    steg_text = HuffmanDecode(codec,extracted_code)

    print(f"steg text for image {i}: {steg_text}")

False
PSNR: 23.04843676466147, SSIM: 0.8719628100364026


SteganographyException: No available slot remaining (image filled)